# Qwen validation and threshold selection

Exact reference-level macro-F0.5, tied-score threshold search, complete candidate-score verification, country reports, persisted policy contracts and locked holdout evaluation are implemented. Threshold selection is a bounded pilot operation: the in-memory evaluator refuses more than 100,000 references or 5 million pairs. Production scoring and submission export stream from disk.

Tune only on threshold_tune. Evaluate holdout once using the locked policy, with no injected positives and the full ground-truth denominator.

In [ ]:
from pathlib import Path
import json

workspace = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "entity_resolution_research").is_dir()
    and (p / "6ab10eb3b23ba_student_resource").is_dir()
)
PROJECT_DIR = workspace / "anshp"
DATA_DIR = workspace / "6ab10eb3b23ba_student_resource" / "student_resource" / "dataset"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
REPORT_DIR = PROJECT_DIR / "reports"
OUTPUT_DIR = PROJECT_DIR / "output"
CONFIG = json.loads((PROJECT_DIR / "config.json").read_text(encoding="utf-8"))
print("Project:", PROJECT_DIR)
print("Project status:", CONFIG["status"])


In [ ]:
from collections import defaultdict
from collections.abc import Mapping
from itertools import groupby
import math


def entity_f05(gold_count, predicted_count, true_positive_count):
    counts = (gold_count, predicted_count, true_positive_count)
    if any(isinstance(x, bool) or not isinstance(x, int) or x < 0 for x in counts):
        raise ValueError("Counts must be nonnegative integers")
    if true_positive_count > min(gold_count, predicted_count):
        raise ValueError("True positives exceed gold or predicted count")
    if gold_count == predicted_count == 0:
        return 1.0
    return 1.25 * true_positive_count / (0.25 * gold_count + predicted_count)


def _checked_id_set(values):
    if isinstance(values, (str, bytes)):
        raise TypeError("Expected an ID collection, not an ID-list string")
    values = list(values)
    if any(not isinstance(x, str) or not x for x in values):
        raise ValueError("IDs must be nonempty strings")
    result = frozenset(values)
    if len(result) != len(values):
        raise ValueError("Duplicate IDs in a match set")
    return result


def _checked_truth(gold_sets):
    if not isinstance(gold_sets, Mapping) or not gold_sets:
        raise ValueError("Complete nonempty reference-to-gold mapping is required")
    if any(not isinstance(qid, str) or not qid for qid in gold_sets):
        raise ValueError("Reference IDs must be nonempty strings")
    return {qid: _checked_id_set(ids) for qid, ids in gold_sets.items()}


def evaluate_match_sets(gold_sets, predictions):
    gold = _checked_truth(gold_sets)
    unknown = set(predictions) - set(gold)
    if unknown:
        raise ValueError("Predictions contain references absent from complete truth")
    entity_scores = []
    total_tp = total_gold = total_pred = n_singleton = correct_singleton = 0
    for qid, actual in gold.items():
        predicted = _checked_id_set(predictions.get(qid, ()))
        tp = len(actual & predicted)
        entity_scores.append(entity_f05(len(actual), len(predicted), tp))
        total_tp += tp
        total_gold += len(actual)
        total_pred += len(predicted)
        if not actual:
            n_singleton += 1
            correct_singleton += int(not predicted)
    return {
        "macro_f05": math.fsum(entity_scores) / len(gold),
        "n_references": len(gold),
        "true_positives": total_tp,
        "gold_pairs": total_gold,
        "predicted_pairs": total_pred,
        "pair_precision": total_tp / total_pred if total_pred else None,
        "pair_recall": total_tp / total_gold if total_gold else None,
        "singleton_accuracy": correct_singleton / n_singleton if n_singleton else None,
    }


def _checked_score_rows(candidate_rows, reference_ids):
    references = set(reference_ids)
    rows, seen = [], set()
    for qid, external_id, raw_score in candidate_rows:
        if qid not in references:
            raise ValueError("Candidate reference is absent from complete truth")
        if not isinstance(external_id, str) or not external_id:
            raise ValueError("External IDs must be nonempty strings")
        key = (qid, external_id)
        if key in seen:
            raise ValueError("Duplicate candidate pair")
        seen.add(key)
        score = float(raw_score)
        if not math.isfinite(score):
            raise ValueError("Qwen scores must be finite")
        rows.append((qid, external_id, score))
    return rows


def optimize_global_threshold(candidate_rows, gold_sets):
    # Reference implementation: O(M log M) time, O(M + N) memory.
    # Production must replace this sort/list with validated external score runs.
    gold = _checked_truth(gold_sets)
    rows = _checked_score_rows(candidate_rows, gold)
    rows.sort(key=lambda row: row[2], reverse=True)
    counts = {qid: [0, 0] for qid in gold}  # predicted, true positive
    total = float(sum(not actual for actual in gold.values()))
    best_score = total / len(gold)
    best_policy = {"mode": "predict_none", "score_space": "raw_yes_no_logit_difference", "threshold": None, "comparator": ">="}
    for threshold, ties in groupby(rows, key=lambda row: row[2]):
        changes = defaultdict(lambda: [0, 0])
        for qid, external_id, _ in ties:
            changes[qid][0] += 1
            changes[qid][1] += int(external_id in gold[qid])
        for qid, (added, hit) in changes.items():
            predicted, tp = counts[qid]
            total -= entity_f05(len(gold[qid]), predicted, tp)
            counts[qid] = [predicted + added, tp + hit]
            total += entity_f05(len(gold[qid]), predicted + added, tp + hit)
        value = total / len(gold)
        # Keep the higher threshold when objective values are numerically tied.
        if value > best_score + 1e-12:
            best_score = value
            best_policy = {"mode": "threshold", "score_space": "raw_yes_no_logit_difference", "threshold": threshold, "comparator": ">="}
    return {"policy": best_policy, "tuning_macro_f05": best_score, "n_references": len(gold), "n_candidates": len(rows)}


def apply_match_policy(candidate_rows, all_reference_ids, policy):
    reference_ids = list(all_reference_ids)
    if len(set(reference_ids)) != len(reference_ids):
        raise ValueError("Duplicate reference IDs")
    if policy.get("score_space") != "raw_yes_no_logit_difference" or policy.get("comparator") != ">=":
        raise ValueError("Decision policy does not match the Qwen score contract")
    mode = policy.get("mode")
    if mode not in {"predict_none", "threshold"}:
        raise ValueError("Unknown decision policy")
    threshold = policy.get("threshold")
    if mode == "threshold" and (threshold is None or not math.isfinite(float(threshold))):
        raise ValueError("A finite fitted threshold is required")
    predicted = {qid: set() for qid in reference_ids}
    for qid, external_id, score in _checked_score_rows(candidate_rows, reference_ids):
        if mode == "threshold" and score >= float(threshold):
            predicted[qid].add(external_id)
    return predicted


In [ ]:
# Small illustrative checks only; these are not dataset performance measurements.
gold = {"S1-a": {"S2-a", "S3-a"}, "S1-b": set(), "S1-c": {"S2-unretrieved"}}
rows = [("S1-a", "S2-a", 3.0), ("S1-a", "S3-a", 3.0), ("S1-b", "S2-wrong", -2.0)]
fit = optimize_global_threshold(rows, gold)
predicted = apply_match_policy(rows, gold, fit["policy"])
assert predicted["S1-a"] == {"S2-a", "S3-a"}
assert predicted["S1-b"] == predicted["S1-c"] == set()
assert abs(evaluate_match_sets(gold, predicted)["macro_f05"] - 2 / 3) < 1e-12
assert entity_f05(0, 0, 0) == 1.0
assert entity_f05(0, 1, 0) == 0.0
assert entity_f05(1, 0, 0) == 0.0
print("Illustrative metric/threshold checks passed; no project metrics measured.")


In [ ]:
# Independently enumerate all threshold policies to check the optimized sweep.
# This regression uses synthetic groups, not project data or measured accuracy.
def run_threshold_regression(cases=1000, seed=2026):
    import random
    import json
    rng = random.Random(seed)
    for case in range(cases):
        gold = {f"q{i}": {f"p{j}" for j in range(rng.randrange(5))}
                for i in range(rng.randrange(1, 8))}
        rows = [(q, f"p{j}", rng.choice([-3., -1., 0., 1., 3.]))
                for q in gold for j in range(7) if rng.random() < 0.6]
        fitted = optimize_global_threshold(rows, gold)
        values = []
        for threshold in [None] + sorted({s for _, _, s in rows}, reverse=True):
            predicted = {q: {p for q2, p, s in rows
                             if q2 == q and threshold is not None and s >= threshold}
                         for q in gold}
            per_reference = []
            for q in gold:
                g, k = len(gold[q]), len(predicted[q])
                tp = len(gold[q] & predicted[q])
                per_reference.append(1.0 if g == k == 0 else 1.25 * tp / (0.25 * g + k))
            values.append(math.fsum(per_reference) / len(gold))
        expected = max(values)
        assert abs(fitted["tuning_macro_f05"] - expected) < 1e-10, case
        policy = json.loads(json.dumps(fitted["policy"], allow_nan=False))
        actual = evaluate_match_sets(gold, apply_match_policy(rows, gold, policy))["macro_f05"]
        assert abs(actual - expected) < 1e-10, case
    for invalid in [[("q", "p", float("nan"))], [("q", "p", float("inf"))],
                    [("q", "p", 1), ("q", "p", 1)], [("unknown", "p", 1)]]:
        try:
            optimize_global_threshold(invalid, {"q": set()})
        except ValueError:
            pass
        else:
            raise AssertionError("Invalid candidate rows accepted")
    print(f"PASS: {cases} brute-force comparisons, policy serialization and input checks")

run_threshold_regression()


## Saved-score evaluation

Complete candidate-set checks, split isolation, country reports, threshold persistence and locked holdout evaluation. The exact threshold sweep is capped at 100,000 references and 5 million pairs; larger evaluations require an external-sort implementation.

In [ ]:
from pathlib import Path
import json
_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else next(
    p / "anshp" for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "anshp" / "config.json").is_file())
_A = {"__file__": str(_ROOT / "00_setup_and_data_audit.ipynb")}
for _c in json.loads((_ROOT / "00_setup_and_data_audit.ipynb").read_text(encoding="utf-8"))["cells"]:
    if _c["cell_type"] == "code" and "er-definitions" in _c.get("metadata", {}).get("tags", []):
        exec("".join(_c["source"]), _A)

import sqlite3
_B, _ = _A["load_helpers"](_ROOT / "02_blocking.ipynb")

def scored_data(score_dir,required_split,max_references=100000,max_pairs=5000000):
    """Bounded pilot evaluator; reject larger inputs rather than exhaust memory."""
    if required_split not in ("early_stop","threshold_tune","holdout"):
        raise ValueError("Evaluation requires a designated non-fit split")
    score_dir=Path(score_dir)
    sm=_A["require_manifest"](score_dir/"manifest.json","scores")
    cd=Path(sm["candidate_dir"]);cm=_A["require_manifest"](cd/"manifest.json","candidates")
    if cm["fingerprint"]!=sm["identity"]["candidates"] or cm["dataset"]!="train":
        raise ValueError("Score/candidate identity mismatch")
    if cm["references"]>max_references or cm["pairs"]>max_pairs:
        raise ValueError("Pilot evaluation limit exceeded; use an external-sort evaluator")
    db=_B["sqlite_readonly"](score_dir/sm["database"])
    try:
        db.execute("ATTACH DATABASE ? AS cdb",((cd/cm["database"]).resolve().as_uri()+"?mode=ro",))
        missing=db.execute("""SELECT reference_id,candidate_id FROM cdb.candidates EXCEPT
                             SELECT reference_id,candidate_id FROM scores LIMIT 1""").fetchone()
        extra=db.execute("""SELECT reference_id,candidate_id FROM scores EXCEPT
                           SELECT reference_id,candidate_id FROM cdb.candidates LIMIT 1""").fetchone()
        if missing or extra:raise ValueError("Scores must equal the exact retained candidate set")
        scope=db.execute("SELECT reference_id,country FROM cdb.query_scope ORDER BY reference_id").fetchall()
        score_rows=db.execute("SELECT reference_id,candidate_id,score FROM scores").fetchall()
    finally:db.close()
    ad=Path(cm["audit_dir"]);audit=_A["require_manifest"](ad/"manifest.json","audit")
    con=_A["connect_database"](ad/audit["database"],read_only=True)
    try:
        scope_path=next(cd/f["path"] for f in cm["files"] if f["path"].startswith("query_scope") and f["path"].endswith(".parquet"))
        con.execute(f"CREATE TEMP VIEW scope AS SELECT * FROM read_parquet({_A['sql_literal'](scope_path)})")
        bad=con.execute("""SELECT q.reference_id FROM scope q LEFT JOIN splits s USING(reference_id)
                           WHERE s.split IS NULL OR s.split!=? LIMIT 1""",[required_split]).fetchone()
        if bad:raise ValueError(f"Scope is not exclusively {required_split}: {bad}")
        gold={q:set() for q,_ in scope}
        for q,c in con.execute("SELECT t.reference_id,t.candidate_id FROM truth t JOIN scope q USING(reference_id)").fetchall():
            gold[q].add(c)
    finally:con.close()
    if not gold:raise ValueError("Empty evaluation scope")
    return sm,cm,scope,score_rows,gold

def policy_contract(sm,cm):
    # The training/test index IDs differ; the retrieval algorithm and cap must agree.
    return {"model":sm["identity"]["model"],"model_config":sm["identity"]["model_config"],
            "qwen_helpers":sm["identity"]["helpers"],
            "retrieval":{k:cm["identity"][k] for k in ("cap","channel_limit","overflow","helpers")}}

def evaluate_scored_set(score_dir,required_split,policy=None):
    sm,cm,scope,rows,gold=scored_data(score_dir,required_split)
    if policy is None:
        if required_split=="holdout":raise ValueError("Holdout cannot tune a threshold")
        selected=optimize_global_threshold(rows,gold)["policy"]
    else:selected=policy
    predictions=apply_match_policy(rows,gold.keys(),selected)
    metrics=evaluate_match_sets(gold,predictions)
    countries={}
    for country in sorted({c for _,c in scope}):
        subset={q:gold[q] for q,c in scope if c==country}
        countries[country]=evaluate_match_sets(subset,{q:predictions[q] for q in subset})
    return {"metrics":metrics,"countries":countries,"policy":selected,
            "contract":policy_contract(sm,cm),"score_fingerprint":sm["fingerprint"],
            "references":len(gold),"candidate_pairs":len(rows)}

def tune_threshold(score_dir,output_dir):
    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)
    result=evaluate_scored_set(score_dir,"threshold_tune")
    path=out/"policy.json";_A["atomic_json"](path,result)
    manifest={"stage":"policy","complete":True,"fingerprint":_A["stable_hash"](result),
              "policy_file":path.name,"files":[_A["file_entry"](path,out)]}
    _A["atomic_json"](out/"manifest.json",manifest)
    return result

def evaluate_holdout(score_dir,policy_dir,report_path):
    policy_dir=Path(policy_dir)
    manifest=_A["require_manifest"](policy_dir/"manifest.json","policy")
    locked=json.loads((policy_dir/manifest["policy_file"]).read_text(encoding="utf-8"))
    result=evaluate_scored_set(score_dir,"holdout",locked["policy"])
    if result["contract"]!=locked["contract"]:
        raise ValueError("Holdout scorer/retriever differs from locked policy")
    _A["atomic_json"](report_path,result)
    return result


In [ ]:
from pathlib import Path
import json
PROJECT_DIR = next(p / "anshp" for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "anshp" / "config.json").is_file())
DATA_DIR = PROJECT_DIR.parent / "6ab10eb3b23ba_student_resource" / "student_resource" / "dataset"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
CONFIG = json.loads((PROJECT_DIR / "config.json").read_text(encoding="utf-8"))

RUN_VALIDATION = False
if RUN_VALIDATION:
    training,_ = _A["load_helpers"](PROJECT_DIR/"03_qwen_training.ipynb")
    model_dir=ARTIFACT_DIR/"model"
    mm=_A["require_manifest"](model_dir/"manifest.json","model")
    _A["load_helpers"](PROJECT_DIR/"03_qwen_training.ipynb",mm["qwen_helpers_sha256"])
    run_config=mm["config"]
    model,tokenizer,yes_no=training["load_qwen"](run_config,adapter_dir=model_dir/mm["adapter"])
    cap=run_config["blocking"]["selected_candidate_cap"] or 40
    # Fixed bounded scopes retain every reference and every final candidate within them.
    for split in ("threshold_tune","holdout"):
        _B["build_candidates"](ARTIFACT_DIR/"index_train",ARTIFACT_DIR/"audit",
            ARTIFACT_DIR/("candidates_"+split),cap=cap,split=split,reference_limit=1000,
            channel_limit=run_config["blocking"]["channel_limit"])
        training["score_candidates"](model,tokenizer,yes_no,ARTIFACT_DIR/("candidates_"+split),
            ARTIFACT_DIR/("scores_"+split),run_config,mm["fingerprint"])
    policy=tune_threshold(ARTIFACT_DIR/"scores_threshold_tune",ARTIFACT_DIR/"policy")
    holdout=evaluate_holdout(ARTIFACT_DIR/"scores_holdout",ARTIFACT_DIR/"policy",
                            PROJECT_DIR/"reports/holdout.json")
    print("Threshold:",policy["policy"])
    print("Held-out metrics:",holdout["metrics"])
